In [2]:
import pandas as pd
import numpy as np
import ast
import math

In [3]:
df = pd.read_csv('Results/ModifiedNarrative-(SV)-bestCandidates.csv')
# df = pd.read_csv('Results/Reddit_candidates-(SV)-NSP-all.csv', converters={"modified_sentences": ast.literal_eval}, chunksize=1000)
# first_chunk = next(df)
# first_chunk.head()

In [41]:
df.iloc[20500]['narrative_modified']

'Therapist trying to make me cry?Long story short...when I was a minor I started having sexual relations with someone 7 years older than me...eventually I ran away with him and I ended up pregnant. He’s terrible person, toxic, bad influence, I was and still am scared of him. 8 years later he decides he wants to establish paternity. I did not resist the perpetrator. My lawyers recommend I get a therapist as I’m going to have to take the stand and tell my story.... So I found a therapist but I can’t help but think every session we have, she’s trying to make me cry. I feel like she wants me to be the victim so I can depend on her. Has anyone else felt like this? I know I’m not a trusting person and overthink everything. I wonder if anyone else has felt the same.'

In [ ]:
def compute_delta_nsp(row):
    """
    delta NSP = mean(prev_prob, next_prob) - orig_prob
    Higher = myth sentence fits more naturally at this insertion point.
    Returns NaN if any required probability is missing.
    """
    prev = row["prev_prob"]
    next_ = row["next_prob"]
    orig = row["orig_prob"]

    # Need at least one of prev/next, and orig for comparison
    if math.isnan(orig):
        return float("nan")
    
    available = [p for p in [prev, next_] if not math.isnan(p)]
    if not available:
        return float("nan")
    
    return sum(available) / len(available) - orig

In [ ]:
df["delta_nsp"] = df.apply(compute_delta_nsp, axis=1)


In [ ]:
# ── Select best candidate per (narrative_idx, myth_type, myth_variation, dose) ─
# Group by narrative + myth configuration, pick insertion with highest delta NSP
best_candidates = (
    df
    .sort_values("delta_nsp", ascending=False)
    .groupby(["narrative_idx", "myth_type", "myth_variation", "dose"], dropna=False)
    .first()
    .reset_index()
)

In [16]:





# ── Rename for summarization pipeline compatibility ───────────────────────────
best_candidates = best_candidates.rename(columns={
    "original_narrative": "narrative_original",
    "modified_narrative": "narrative_modified",
})

# Add a clean unique ID for the summarization pipeline resume logic
best_candidates["id"] = range(len(best_candidates))

# ── Report ────────────────────────────────────────────────────────────────────
print(f"Total candidates:      {len(df)}")
print(f"Best candidates kept:  {len(best_candidates)}")
print(f"Delta NSP stats:\n{best_candidates['delta_nsp'].describe().round(4)}")
print(f"\nRows with NaN delta NSP: {best_candidates['delta_nsp'].isna().sum()}")

# ── Save ──────────────────────────────────────────────────────────────────────


pandas.io.parsers.readers.TextFileReader

In [ ]:
best_candidates.to_csv(
    "Results/ModifiedNarrativeCandidates-(SV)-bestCandidates.csv",
    index=False
)